In [1]:
import json

# load json pages
with open("pages.json", "r") as f:
    pages = json.load(f)


# load m_records
with open("MTI/m_records.json", "r") as f:
    m_records = json.load(f)


mpages = {}
for mrecord in m_records:
    if mrecord["pageType"] == 0xfd:
        page_num = mrecord["page"][0] * 9 + mrecord["page"][1]
        if page_num not in mpages:
            mpages[page_num] = {}

        pos = f"{mrecord['pos'][0]}-{mrecord['pos'][1]}"
        mpages[page_num][pos] = mrecord

n_matched = 0
page2mpage = {}
for page_num, page in pages.items():

    max_match = 0
    best_match = None
    for mpage_num, mpage in mpages.items():

        match_count = 0
        for button in page["buttons"]:
            pos = f"{button['position'][1]}-{button['position'][0]}"
            button['pos'] = pos
            if pos in mpage:
                mbutton = mpage[pos]
                l = mbutton.get("label", "").strip()
                ml = button.get("label", "").strip()
                m = mbutton.get("message", "").strip()
                mm = button.get("message", "").strip()
                if (l == ml and l != "") or (m == mm and m != ""):
                    match_count += 1
    
        if match_count > max_match:
            max_match = match_count
            best_match = mpage_num

    if best_match is not None:
        page2mpage[page_num] = best_match
        print(f"Page {page_num} matched to MPage {best_match} with {max_match}/{len(page['buttons'])} matches")
        n_matched += 1


Page 1 matched to MPage 0 with 84/84 matches
Page 2 matched to MPage 45 with 17/18 matches
Page 3 matched to MPage 54 with 21/22 matches
Page 4 matched to MPage 63 with 6/7 matches
Page 5 matched to MPage 72 with 6/7 matches
Page 6 matched to MPage 81 with 6/7 matches
Page 7 matched to MPage 90 with 6/7 matches
Page 60 matched to MPage 567 with 36/38 matches
Page 61 matched to MPage 1 with 6/7 matches
Page 62 matched to MPage 10 with 5/6 matches
Page 63 matched to MPage 19 with 5/6 matches
Page 64 matched to MPage 28 with 9/10 matches
Page 65 matched to MPage 37 with 6/7 matches
Page 66 matched to MPage 46 with 8/9 matches
Page 67 matched to MPage 55 with 6/7 matches
Page 446 matched to MPage 16 with 29/33 matches
Page 127 matched to MPage 20 with 28/29 matches
Page 128 matched to MPage 29 with 22/23 matches
Page 528 matched to MPage 107 with 7/9 matches
Page 529 matched to MPage 98 with 6/7 matches
Page 10 matched to MPage 117 with 37/39 matches
Page 11 matched to MPage 126 with 15/16

In [2]:
def build_page_tree(pages):
    visted = set()
    page2node = {}
   
    def helper(page_num, parent=None, linking_button={}):
        node = {"children": [], "parent": parent, "linking_button": linking_button, "page_num": page_num}
        node_page = pages.get(str(page_num))
        for button in node_page["buttons"]:
            linked_page = button.get("linked_page", None)
            if linked_page and (linked_page not in visted):
                    visted.add(linked_page)
                    child_node = helper(linked_page, parent=node, linking_button=button)
                    node["children"].append(child_node)
        page2node[str(page_num)] = node
        return node
    
    visted.add("1")
    root = helper("1")
    return root, page2node

            
def get_path_2_root(node):
    path = []
    while node:
        # path.append(str(node.get("page_num")))
        button = node.get("linking_button", None)
        if button:
            label = button.get("label", "")
            pos = button.get("pos", "")
            path.append(f"{label}({pos})")
        node = node["parent"]
    return path[::-1]



page_tree, page2node = build_page_tree(pages)
for page_num, node in page2node.items():
    path = " -> ".join(get_path_2_root(node))
    print(f"Page {page_num} path to root: home -> {path}")


Page 2 path to root: home -> me(1-0)
Page 4 path to root: home -> my(1-1) -> COPY(3-3)
Page 5 path to root: home -> my(1-1) -> MAIL(5-7)
Page 6 path to root: home -> my(1-1) -> PRINT(5-1)
Page 7 path to root: home -> my(1-1) -> SPELL(3-11)
Page 3 path to root: home -> my(1-1)
Page 61 path to root: home -> wear(1-2) -> TIE(3-10)
Page 62 path to root: home -> wear(1-2) -> TIGHT(5-6)
Page 63 path to root: home -> wear(1-2) -> LOOSE(6-7)
Page 64 path to root: home -> wear(1-2) -> ZIP(4-11)
Page 65 path to root: home -> wear(1-2) -> DRESS(5-8)
Page 66 path to root: home -> wear(1-2) -> N'T(6-1)
Page 67 path to root: home -> wear(1-2) -> BUTTON(3-11)
Page 446 path to root: home -> wear(1-2) -> CLOTHING(0-1)
Page 60 path to root: home -> wear(1-2)
Page 128 path to root: home -> am(1-3) -> N'T(6-1)
Page 528 path to root: home -> am(1-3) -> BEHAVE(3-6)
Page 529 path to root: home -> am(1-3) -> BECOME(4-4)
Page 127 path to root: home -> am(1-3)
Page 10 path to root: home -> please(1-4)
Page 11 p

In [3]:
page1 = pages["1"]["buttons"]
mpage1 = mpages[page2mpage["1"]]



buttons_needing_icons = 0
buttons_linked_to_icons = 0

pages_with_missing_icons = set()
missing_on_page = {}
for page_num, page in pages.items():
    if page_num in page2mpage:
        mpage = mpages[page2mpage[page_num]]

        # for each button with a symbol link
        for button in page["buttons"]:
            if button.get("symbol_link", None) is not None:
                
                # Get the button pos
                pos = button['pos']
                buttons_needing_icons += 1
                label = button.get("label", button.get("message", ""))

                # If the pos is in the mpage, get the icon and link it to the button
                if pos in mpage:
                    mbutton = mpage[pos]
                    icon = mbutton.get("iconName", None)
                    if icon is not None:
                        button["icon_file"] = icon
                        buttons_linked_to_icons += 1
                    else:
                        pages_with_missing_icons.add(page_num)
                        if page_num not in missing_on_page:
                            missing_on_page[page_num] = []
                        missing_on_page[page_num].append(f"{label}({pos})")
                        # missing_icon_buttons.append((page_num, pos, label))
                        # print(f"Button ({pos}, {label}) needs an icon but no iconName found at pos {pos} in page {page_num} MPage {page2mpage[page_num]}")
                    
                else:
                    pages_with_missing_icons.add(page_num)
                    if page_num not in missing_on_page:
                        missing_on_page[page_num] = []
                    missing_on_page[page_num].append(f"{label}({pos})")
                    # missing_icon_buttons.append((page_num, pos, label))
                    # print(f"Button ({pos}, {label}) needs an icon but no MRecord found at pos {pos} in page {page_num} MPage {page2mpage[page_num]}")
                
    


print(f"Matched {n_matched} pages out of {len(pages)}")
print(f"Linked {buttons_linked_to_icons} buttons to icons out of {buttons_needing_icons} needing icons")

for page_num in pages_with_missing_icons:
    node = page2node.get(page_num, None)
    path = "/".join(get_path_2_root(node))
    missing = ", ".join(missing_on_page.get(page_num, []))
    print(f"home/{path} = {page_num} has buttons needing icons: {missing}")

# # save updated pages with icon links
# with open("pages_with_icons.json", "w") as f:
#     json.dump(pages, f, indent=2)

Matched 525 pages out of 525
Linked 6411 buttons to icons out of 6467 needing icons
home/off(0-10) = 400 has buttons needing icons: bald(5-4)
home/have(3-6)/STEAL(3-7) = 538 has buttons needing icons: stolen(2-6)
home/play(3-3)/TICKLE(3-10) = 533 has buttons needing icons: ticklish(2-8)
home/get(4-8)/TERRAIN(0-4) = 510 has buttons needing icons: cactus(4-4)
home/feel(3-7)/PROUD(3-2) = 235 has buttons needing icons: pride(1-9)
home/play(3-3) = 42 has buttons needing icons: SPIN(6-10), TICKLE(3-10)
home/SPELL/NUM(2-8)/PHONICS PAGE(0-3) = 565 has buttons needing icons: ch(0-3), sh(0-4), th(0-5), TH(0-6), wh(0-7), f(1-6), SHORT(1-7), LONG(1-8), k(2-6), SHORT(2-7), LONG(2-8), LONG(3-8), DEL WORD(4-0), LONG(5-8), (5-0), LONG(4-8), SILENT(6-8), br(0-2), tr(3-0), sp(6-1), st(6-2)
home/play(3-3)/TOYS(0-3) = 511 has buttons needing icons: tricycle(4-10), wagon(5-1)
home/live(5-7)/DWELLING(0-11) = 531 has buttons needing icons: igloo(0-10)
home/end(2-11)/GOVERN(5-7) = 207 has buttons needing icon

In [34]:
print(get_page_path("565", pages))

None


In [26]:
!node upload-icons.js

[dotenv@17.2.3] injecting env (17) from ../Firebase Functions/function-tests/test.env -- tip: ✅ audit secrets and track compliance: https://dotenvx.com/ops
Found 1801 unique icons to upload.


In [ ]:
# load json pages
with open("pages_with_icons_updated.json", "r") as f:
    pages = json.load(f)


for page_num, page in pages.items():
    for button in page["buttons"]:
        action = button.get("actions", "")
        if "C10" in action:
            message = button.get("message", button.get("label", ""))
            if len(message) > 1:
                if message and not message.endswith(" "):
                    button["message"] = message + " "

# save updated pages with icon links
with open("lamp_pages.json", "w") as f:
    json.dump(pages, f, indent=2)
